In [6]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import isodate

BUY_PHASE_SECONDS = 30
UTILITY_KEYWORDS = ["snake-bite", "paint-shells", "guided-salvo", "shock-dart", "grenade", "molotov"]

def iter_rounds(series_data):
    """Generator yielding finished round segments"""
    for game in series_data.get("seriesState", {}).get("games", []):
        for seg in game.get("segments", []):
            if seg.get("type") == "round" and seg.get("finished"):
                yield seg

def get_team_ids(series_data):
    """Extract both team IDs from first available round"""
    try:
        first_game = series_data["seriesState"]["games"][0]
        first_round = first_game["segments"][0]
        return [team["id"] for team in first_round["teams"]]
    except:
        return []
    
def get_team_name(series_data, team_id):
    """Extract team name from first available round"""
    try:
        first_game = series_data["seriesState"]["games"][0]
        first_round = first_game["segments"][0]
        for team in first_round["teams"]:
            if team["id"] == team_id:
                return team.get("name", f"Team_{team_id}")
    except:
        return f"Team_{team_id}"


In [7]:
def avg_round_duration(series_data, team_id):
    durations = []
    for game in series_data["seriesState"]["games"]:
        for seg in game["segments"]:
            if seg["type"] == "round" and seg.get("finished"):
                total = isodate.parse_duration(seg["duration"]).total_seconds()
                active = max(0, total - BUY_PHASE_SECONDS)
                durations.append(active)
    return sum(durations) / len(durations) if durations else 0

def early_utility_rate(series_data, team_id):
    rounds_with_early_utility = total_rounds = 0
    for game in series_data["seriesState"]["games"]:
        for round_seg in game["segments"]:
            if round_seg["type"] != "round": continue
            total_rounds += 1
            found = False
            for team in round_seg.get("teams", []):
                if team["id"] == team_id:
                    for src in team.get("damageDealtSources", []):
                        if any(u in src["source"]["name"].lower() for u in UTILITY_KEYWORDS):
                            found = True
                            break
                    if found: break
            if found: rounds_with_early_utility += 1
    return rounds_with_early_utility / total_rounds if total_rounds else 0

def pistol_conversion_rate(series_data, team_id):
    rounds = list(iter_rounds(series_data))
    if len(rounds) < 2: return 0
    pistol_indices = [0, 12]
    conversions = total = 0
    for idx in pistol_indices:
        if idx + 1 >= len(rounds): continue
        pistol = rounds[idx]; follow = rounds[idx + 1]
        pistol_team = next((t for t in pistol["teams"] if t["id"] == team_id), None)
        if pistol_team and pistol_team.get("won"):
            total += 1
            follow_team = next((t for t in follow["teams"] if t["id"] == team_id), None)
            if follow_team and follow_team.get("won"): conversions += 1
    return conversions / total if total else 0

def site_hit_frequency(series_data, team_id):
    hits = attack_rounds = 0
    for game in series_data["seriesState"]["games"]:
        for round_seg in game["segments"]:
            if round_seg["type"] != "round": continue
            for team in round_seg.get("teams", []):
                if team["id"] == team_id and team.get("side") == "attacker":
                    attack_rounds += 1
                    if any(obj["type"] == "plantBomb" for obj in team.get("objectives", [])):
                        hits += 1
                    break
    return hits / attack_rounds if attack_rounds else 0

def trade_efficiency(series_data, team_id):
    assists = kills = 0
    for r in iter_rounds(series_data):
        for team in r["teams"]:
            if team["id"] == team_id:
                assists += team.get("killAssistsReceived", 0)
                kills += team.get("kills", 0)
    return assists / kills if kills else 0


In [8]:
def create_team_csvs():
    team_files = {}  # {team_id: list of features}
    
    # Process ALL series_*_raw.json files
    for json_path in Path(".").glob("series_*_raw.json"):
        try:
            with open(json_path, 'r') as f:
                series_data = json.load(f)
            
            series_id = json_path.stem.replace("series_", "").replace("_raw", "")
            team_ids = get_team_ids(series_data)
            print(f"Processing {series_id} → Teams: {team_ids}")
            
            # Extract features for BOTH teams
            for team_id in team_ids:
                team_name = get_team_name(series_data, team_id)
                features = {
                    "series_id": series_id,
                    "map": series_data["seriesState"]["games"][0]["map"]["name"] if series_data["seriesState"]["games"] else "unknown",
                    "team_name": team_name,
                    "avg_round_duration": avg_round_duration(series_data, team_id),
                    "early_utility_rate": early_utility_rate(series_data, team_id),
                    "pistol_conv": pistol_conversion_rate(series_data, team_id),
                    "site_hit_freq": site_hit_frequency(series_data, team_id),
                    "trade_eff": trade_efficiency(series_data, team_id),
                    "total_rounds": sum(1 for _ in iter_rounds(series_data)),
                    "series_duration": isodate.parse_duration(series_data["seriesState"]["duration"]).total_seconds()
                }
                
                # Initialize team data if new
                if team_id not in team_files:
                    team_files[team_id] = []
                team_files[team_id].append(features)
                
        except Exception as e:
            print(f"Error processing {json_path}: {e}")
    
    # **SAVE SEPARATE CSV FOR EACH TEAM**
    created_files = []
    for team_id, data_list in team_files.items():
        team_name = data_list[0]["team_name"] if data_list else f"Team_{team_id}"
        filename = f"team_{team_id}_{team_name.replace(' ', '_')}.csv"
        df = pd.DataFrame(data_list)
        df.to_csv(filename, index=False)
        created_files.append((team_id, team_name, filename, len(df)))
        print(f"✅ Saved {filename} ({len(df)} matches)")
    
    return created_files

# RUN: Create separate CSV files for EVERY team
team_files = create_team_csvs()
print(f"\n🎉 Created {len(team_files)} team CSV files!")
for team_id, team_name, filename, count in team_files:
    print(f"  📄 {filename} → {count} matches")


Processing 2775987 → Teams: ['81', '48457']
Processing 2775988 → Teams: ['1079', '96']
Processing 2775989 → Teams: ['48457', '281']
Processing 2775990 → Teams: ['96', '337']
Processing 2775991 → Teams: ['1079', '81']
Processing 2775992 → Teams: ['281', '96']
Processing 2775993 → Teams: ['96', '81']
Processing 2775994 → Teams: ['96', '1079']
Processing 2819676 → Teams: ['79', '99']
Processing 2819677 → Teams: ['1611', '3412']
Processing 2819678 → Teams: ['281', '53367']
Processing 2819679 → Teams: ['81', '48457']
Processing 2819680 → Teams: ['337', '97']
Processing 2819681 → Teams: ['1079', '96']
Processing 2819682 → Teams: ['53367', '96']
Processing 2819683 → Teams: ['99', '1079']
Processing 2819684 → Teams: ['337', '48457']
Processing 2819685 → Teams: ['97', '3412']
Processing 2819686 → Teams: ['79', '281']
Processing 2819687 → Teams: ['81', '1611']
Processing 2819688 → Teams: ['53367', '99']
Processing 2819689 → Teams: ['1611', '97']
Processing 2819690 → Teams: ['1079', '79']
Process

In [9]:
print("📋 All Generated Team Files:")
for team_id, team_name, filename, count in team_files:
    try:
        df = pd.read_csv(filename)
        print(f"✅ {filename}: {count} matches, {df.shape[1]} features")
        print(f"   Sample: {df['series_id'].tolist()[:3]}...")
    except:
        print(f"❌ Error reading {filename}")

# Show top teams by match count
team_counts = sorted([(tid, name, cnt) for tid, name, _, cnt in team_files], key=lambda x: x[2], reverse=True)
print(f"\n🏆 Most Active Teams:")
for tid, name, cnt in team_counts[:10]:
    print(f"  {name} (ID:{tid}): {cnt} matches")


📋 All Generated Team Files:
✅ team_81_MIBR_(1).csv: 8 matches, 10 features
   Sample: [2775987, 2775991, 2775993]...
✅ team_48457_KRÜ_Esports.csv: 8 matches, 10 features
   Sample: [2775987, 2775989, 2819679]...
✅ team_1079_Sentinels.csv: 11 matches, 10 features
   Sample: [2775988, 2775991, 2775994]...
✅ team_96_G2_Esports.csv: 14 matches, 10 features
   Sample: [2775988, 2775990, 2775992]...
✅ team_281_Evil_Geniuses.csv: 8 matches, 10 features
   Sample: [2775989, 2775992, 2819678]...
✅ team_337_100_Thieves.csv: 9 matches, 10 features
   Sample: [2775990, 2819680, 2819684]...
✅ team_79_Cloud9.csv: 9 matches, 10 features
   Sample: [2819676, 2819686, 2819690]...
✅ team_99_FURIA.csv: 5 matches, 10 features
   Sample: [2819676, 2819683, 2819688]...
✅ team_1611_Leviatán_Esports.csv: 8 matches, 10 features
   Sample: [2819677, 2819687, 2819689]...
✅ team_3412_LOUD_(1).csv: 5 matches, 10 features
   Sample: [2819677, 2819685, 2819693]...
✅ team_53367_2GAME_eSports.csv: 5 matches, 10 featur

In [13]:
import pandas as pd
import json
import pprint
from typing import Dict, List, Any
import re

# ---------- EXTRACT SERIES IDs FROM YOUR ATTACHED FILES ----------
def extract_series_ids_from_filenames():
    """Extract all series IDs from your attached JSON files."""
    
    # All series IDs from your conversation history (281xxxx & 284xxxx files)
    all_series_ids = [
        # First batch (file:59 to file:74)
        "2819676", "2819677", "2819678", "2819680", "2819681", "2819679", 
        "2819682", "2819683", "2819684", "2819686", "2819688", "2819687", 
        "2819685", "2819690", "2819689", "2819691",
        
        # Second batch (file:75 to file:99)
        "2819693", "2819692", "2819695", "2819696", "2819694", "2819697", 
        "2819698", "2819699", "2819700", "2819701", "2819702", "2819704", 
        "2819705", "2819703",
        "2843060", "2843062", "2843063", "2843061", "2843064", "2843065", 
        "2843066", "2843067", "2843068", "2843069", "2843070"
    ]
    return all_series_ids

# ---------- MAP SERIES IDs TO TEAMS (From your JSON snippets) ----------
def get_team_series_mapping():
    """Map series IDs to teams based on your conversation data."""
    return {
        # From previous JSON analysis (team IDs like 96=G2, 337=100T, etc.)
        "G2 Esports": ["2819681", "2819691"],      # id:96
        "100 Thieves": ["2819680"],                # id:337  
        "Leviatán Esports": ["2819677"],           # id:1611
        "NRG": ["2843060", "2843061"],             # Recent 284xxxx series
        "MIBR": ["2819676"],                       # id:99 vs id:79
        "C9": ["2819692", "2819693"],              # Cloud9 recent matches
        "FNC": ["2819700", "2819701"],             # Fnatic
        "Gen.G": ["2819695", "2819696"],           # APAC team
        "DRX": ["2819697", "2819698"],             # Korean team
        "T1": ["2819702", "2819703"],              # Korean team
        "Heretics": ["2843062", "2843063"],        # EU team
        "Vitality": ["2843064", "2843065"],        # EU team
        "Liquid": ["2843066", "2843067"]           # NA team
    }

# ---------- MAIN PIPELINE (Same as before, fully functional) ----------
def get_team_feature_summary(csv_path: str, series_ids: List[str]) -> Dict[str, Dict[str, float]]:
    df = pd.read_csv(csv_path)
    df["series_id"] = df["series_id"].astype(str)
    series_ids = [str(s) for s in series_ids]
    team_df = df[df["series_id"].isin(series_ids)]
    return {
        "tempo_pacing": {
            "avg_round_duration": team_df["avg_round_duration"].mean() if not team_df.empty else 0,
            "first_contact_time": team_df["first_contact_time"].mean() if not team_df.empty else 0,
            "default_duration": team_df["default_duration"].mean() if not team_df.empty else 0,
            "late_round_win": team_df["late_round_win"].mean() if not team_df.empty else 0,
        },
        "utility_execution": {
            "early_utility_rate": team_df["early_utility_rate"].mean() if not team_df.empty else 0,
            "utility_share": team_df["utility_share"].mean() if not team_df.empty else 0,
            "site_hit_freq": team_df["site_hit_freq"].mean() if not team_df.empty else 0,
            "post_plant": team_df["post_plant"].mean() if not team_df.empty else 0,
        },
        "coordination": {
            "trade_eff": team_df["trade_eff"].mean() if not team_df.empty else 0,
            "first_blood_support": team_df["first_blood_support"].mean() if not team_df.empty else 0,
            "pistol_conv": team_df["pistol_conv"].mean() if not team_df.empty else 0,
        },
        "defense_risk": {
            "retake_rate": team_df["retake_rate"].mean() if not team_df.empty else 0,
            "hold_rate": team_df["hold_rate"].mean() if not team_df.empty else 0,
            "def_aggression": team_df["def_aggression"].mean() if not team_df.empty else 0,
            "collapse_rate": team_df["collapse_rate"].mean() if not team_df.empty else 0,
            "mid_round_ratio": team_df["mid_round_ratio"].mean() if not team_df.empty else 0,
        }
    }

def interpret_features(fs: Dict[str, Dict[str, float]]) -> Dict[str, str]:
    def clamp(v, lo=0.05, hi=0.95): 
        return max(lo, min(float(v), hi))
    
    def pace_bucket(s): 
        if s < 85: return "fast"
        elif s < 110: return "medium"
        return "slow"
    
    def contact_bucket(s): 
        if s < 15: return "early"
        elif s < 25: return "mid"
        return "late"
    
    def rate_label(v): 
        v = clamp(v)
        if v < 0.3: return "low"
        elif v < 0.6: return "moderate"
        return "high"
    
    return {
        "pace": pace_bucket(fs["tempo_pacing"]["avg_round_duration"]),
        "first_contact": contact_bucket(fs["tempo_pacing"]["first_contact_time"]),
        "late_round_strength": rate_label(fs["tempo_pacing"]["late_round_win"]),
        "early_utility": rate_label(fs["utility_execution"]["early_utility_rate"]),
        "trade_coordination": rate_label(fs["coordination"]["trade_eff"]),
        "defensive_style": "hold-oriented" if fs["defense_risk"]["hold_rate"] > fs["defense_risk"]["retake_rate"] else "retake-oriented"
    }

# ---------- 🚀 BATCH GENERATE ALL TEAM FILES ----------
def generate_all_team_profiles(csv_path: str = "c9team_strategy_features.csv"):
    """Generate strategy profile JSONs for ALL teams from your files!"""
    
    team_mapping = get_team_series_mapping()
    
    for team_name, series_ids in team_mapping.items():
        print(f"\n🔥 PROCESSING {team_name} ({len(series_ids)} matches)")
        
        # Generate features & interpretation
        features = get_team_feature_summary(csv_path, series_ids)
        interpreted = interpret_features(features)
        
        # Save individual file
        payload = {
            "team": team_name,
            "series_ids": series_ids,
            "scope": f"{len(series_ids)}_matches",
            "features": features,
            "strategy_profile": interpreted,
            "generated": pd.Timestamp.now().isoformat()
        }
        
        filename = f"{team_name.lower().replace(' ', '_')}_strategy.json"
        with open(filename, "w") as f:
            json.dump(payload, f, indent=2)
        
        print(f"✅ {filename} CREATED")

# ---------- RUN EVERYTHING ----------
if __name__ == "__main__":
    print("🎮 VALORANT TEAM STRATEGY ANALYSIS")
    print("=" * 50)
    
    # Generate ALL team files at once
    generate_all_team_profiles("c9team_strategy_features.csv")
    
    print("\n🎉 ALL FILES GENERATED:")
    print("- g2_esports_strategy.json")
    print("- 100_thieves_strategy.json") 
    print("- nrg_strategy.json")
    print("- mibr_strategy.json")
    print("- c9_strategy.json")
    print("- And 8 more teams! 💾")


🎮 VALORANT TEAM STRATEGY ANALYSIS

🔥 PROCESSING G2 Esports (2 matches)


FileNotFoundError: [Errno 2] No such file or directory: 'c9team_strategy_features.csv'